# Phase 3 – Model Training (SVM & Decision Tree)

Train and tune SVM and Decision Tree classifiers using the pre-extracted image features.

## Section 1: Load & clean dataset
Load feature data (with `label`) generated in Phase 2.

In [12]:

from pathlib import Path
import logging
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    force=True,
)

data_path = Path("feature2.csv")
if not data_path.exists():
    fallback_path = Path("feature2.csv")
    if fallback_path.exists():
        logging.info("feature.csv not found. Falling back to features.csv.")
        data_path = fallback_path
    else:
        raise FileNotFoundError("feature.csv not found (also checked features.csv).")

logging.info("Loading dataset from %s", data_path)
df = pd.read_csv(data_path)
logging.info("Loaded dataset shape: %s", df.shape)
logging.info("Columns: %s", df.columns.tolist())

df.head()


2025-12-03 20:48:41,955 [INFO] Loading dataset from feature2.csv
2025-12-03 20:48:42,062 [INFO] Loaded dataset shape: (26179, 40)
2025-12-03 20:48:42,062 [INFO] Columns: ['filename', 'class', 'label', 'width', 'height', 'aspect_ratio', 'mean_red', 'mean_green', 'mean_blue', 'std_red', 'std_green', 'std_blue', 'mean_hue', 'mean_saturation', 'mean_value', 'std_hue', 'std_saturation', 'std_value', 'brightness', 'contrast_simple', 'mean_intensity', 'std_intensity', 'min_intensity', 'max_intensity', 'range_intensity', 'skewness', 'kurtosis', 'entropy', 'contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM', 'edge_mean', 'edge_std', 'edge_density', 'contour_area', 'contour_perimeter', 'compactness']


,filename,class,label,width,height,aspect_ratio,mean_red,mean_green,mean_blue,std_red,...,homogeneity,energy,correlation,ASM,edge_mean,edge_std,edge_density,contour_area,contour_perimeter,compactness
0,butterfly (1).jpeg,butterfly,0,300.0,225.0,1.333333,151.847351,147.497314,155.843201,63.941635,...,0.362407,0.044357,0.935082,0.001978,378.291992,215.509415,0.069519,271.0,719.837646,152.156311
1,butterfly (1).jpg,butterfly,0,426.0,640.0,0.665625,165.210327,153.734741,122.301636,31.479156,...,0.318872,0.034369,0.692184,0.001188,281.857483,118.494362,0.113159,460.5,598.055908,61.807911
2,butterfly (1).png,butterfly,0,640.0,533.0,1.200750,31.207031,26.439392,28.153381,73.013542,...,0.729874,0.690564,0.786207,0.476904,430.259155,172.207748,0.117432,163.5,143.639603,10.042014
3,butterfly (10).jpeg,butterfly,0,300.0,225.0,1.333333,55.270874,60.944275,34.758972,44.147003,...,0.156679,0.019737,0.742181,0.000391,238.287582,135.726318,0.161804,221.0,1074.915894,416.051239
4,butterfly (10).jpg,butterfly,0,640.0,480.0,1.333333,110.409607,114.204834,79.344299,31.464451,...,0.284896,0.035069,0.804062,0.001241,238.503281,138.534393,0.083801,201.0,819.317871,265.765717


In [13]:
import pandas as pd

df = pd.read_csv("feature2.csv")

# ساخت ستون برچسب عددی از روی نام کلاس
df["label"] = df["class"].astype("category").cat.codes

target_column = "label"
if target_column not in df.columns:
    raise KeyError(f"Target column '{target_column}' not found in the dataset.")

y = df[target_column]
X_raw = df.drop(columns=[target_column])

non_numeric_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
if non_numeric_cols:
    logging.info("Dropping non-numeric columns: %s", non_numeric_cols)
else:
    logging.info("No non-numeric columns detected among features.")

X = df.select_dtypes(include=["int64", "float64"]).drop(columns=[target_column], errors="ignore")

if X.empty:
    raise ValueError("No numeric feature columns remain after dropping non-numeric columns.")

logging.info("Final feature matrix shape after cleaning: %s", X.shape)


2025-12-03 20:48:42,167 [INFO] Dropping non-numeric columns: ['filename', 'class']
2025-12-03 20:48:42,171 [INFO] Final feature matrix shape after cleaning: (26179, 37)


## Section 2: Drop non-numeric columns
Remove all non-numeric feature columns (e.g., filenames/paths) before training.

In [14]:

target_column = "label"
if target_column not in df.columns:
    raise KeyError(f"Target column '{target_column}' not found in the dataset.")

y = df[target_column]
X_raw = df.drop(columns=[target_column])

non_numeric_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
if non_numeric_cols:
    logging.info("Dropping non-numeric columns: %s", non_numeric_cols)
else:
    logging.info("No non-numeric columns detected among features.")

X = df.select_dtypes(include=["int64", "float64"]).drop(columns=[target_column], errors="ignore")

if X.empty:
    raise ValueError("No numeric feature columns remain after dropping non-numeric columns.")

logging.info("Final feature matrix shape after cleaning: %s", X.shape)


2025-12-03 20:48:42,182 [INFO] Dropping non-numeric columns: ['filename', 'class']
2025-12-03 20:48:42,186 [INFO] Final feature matrix shape after cleaning: (26179, 37)


## Section 3: Train/Test split
Use stratified split so label proportions are preserved.

In [15]:

labels = list(pd.unique(y))
try:
    labels = sorted(labels)
except TypeError:
    labels = sorted(labels, key=lambda val: str(val))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

logging.info("Train shape: %s; Test shape: %s", X_train.shape, X_test.shape)
logging.info("Train label distribution: %s", y_train.value_counts(normalize=True).round(3).to_dict())
logging.info("Test label distribution: %s", y_test.value_counts(normalize=True).round(3).to_dict())

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logging.info("Using StratifiedKFold (n_splits=5, shuffle=True, random_state=42) for all grid searches.")


2025-12-03 20:48:42,200 [INFO] Train shape: (20943, 37); Test shape: (5236, 37)
2025-12-03 20:48:42,204 [INFO] Train label distribution: {4: 0.186, 8: 0.184, 2: 0.118, 6: 0.1, 0: 0.081, 3: 0.071, 9: 0.071, 7: 0.07, 1: 0.064, 5: 0.055}
2025-12-03 20:48:42,204 [INFO] Test label distribution: {4: 0.186, 8: 0.184, 2: 0.118, 6: 0.1, 0: 0.081, 3: 0.071, 9: 0.071, 7: 0.07, 1: 0.064, 5: 0.055}
2025-12-03 20:48:42,204 [INFO] Using StratifiedKFold (n_splits=5, shuffle=True, random_state=42) for all grid searches.


## Section 4: SVM training + tuning + evaluation

In [16]:
import logging
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# فرض: X_train, X_test, y_train, y_test, labels قبلاً تعریف شده‌اند

# -------------------------------
# 1) نرمال‌سازی فیچرها
# -------------------------------
logging.info("Scaling features for baseline SVM with StandardScaler...")

scaler_baseline = StandardScaler()
X_train_scaled = scaler_baseline.fit_transform(X_train)
X_test_scaled = scaler_baseline.transform(X_test)

# -------------------------------
# 2) تنظیم دستی C و gamma
#    این مقادیر را هر طور خواستی عوض کن
# -------------------------------
C_value = 5       # مثلا 10
gamma_value = 0.1   # مثلا 0.01

logging.info(
    "Training baseline SVM with manual hyperparameters: C=%.4f, gamma=%.4f, kernel='rbf'",
    C_value,
    gamma_value,
)

svm_baseline = SVC(
    C=C_value,
    gamma=gamma_value,
    kernel="rbf",
    random_state=42
)

# -------------------------------
# 3) آموزش مدل روی داده‌های نرمال‌شده
# -------------------------------
svm_baseline.fit(X_train_scaled, y_train)

# -------------------------------
# 4) پیش‌بینی و دقت روی Train و Test
# -------------------------------
y_train_pred = svm_baseline.predict(X_train_scaled)
y_test_pred = svm_baseline.predict(X_test_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

logging.info("Baseline SVM (manual C,gamma) TRAIN accuracy: %.4f", train_acc)
logging.info("Baseline SVM (manual C,gamma) TEST  accuracy: %.4f", test_acc)

# -------------------------------
# 5) گزارش و ماتریس سردرگمی
# -------------------------------
svm_baseline_report = classification_report(y_test, y_test_pred)
logging.info("Baseline SVM classification report:\n%s", svm_baseline_report)

svm_baseline_cm = confusion_matrix(y_test, y_test_pred, labels=labels)
svm_baseline_cm_df = pd.DataFrame(
    svm_baseline_cm,
    index=[f"true_{lbl}" for lbl in labels],
    columns=[f"pred_{lbl}" for lbl in labels]
)
logging.info("Baseline SVM confusion matrix:\n%s", svm_baseline_cm_df)


2025-12-03 20:48:42,212 [INFO] Scaling features for baseline SVM with StandardScaler...
2025-12-03 20:48:42,221 [INFO] Training baseline SVM with manual hyperparameters: C=5.0000, gamma=0.1000, kernel='rbf'
2025-12-03 20:49:39,536 [INFO] Baseline SVM (manual C,gamma) TRAIN accuracy: 0.8961
2025-12-03 20:49:39,536 [INFO] Baseline SVM (manual C,gamma) TEST  accuracy: 0.4908
2025-12-03 20:49:39,544 [INFO] Baseline SVM classification report:
              precision    recall  f1-score   support

           0       0.63      0.65      0.64       422
           1       0.39      0.32      0.35       334
           2       0.48      0.55      0.52       620
           3       0.38      0.38      0.38       373
           4       0.43      0.51      0.46       973
           5       0.41      0.30      0.35       289
           6       0.42      0.39      0.41       525
           7       0.45      0.32      0.38       364
           8       0.64      0.70      0.67       964
           9     

In [17]:
import logging
from scipy.stats import loguniform
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

# -------------------------------
# Logging config
# -------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,
)

# -------------------------------
# Pipeline: StandardScaler + SVC
# -------------------------------
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),      # نرمال‌سازی همه فیچرها
    ("svm", SVC(random_state=42)),     # خود SVM
])

# -------------------------------
# Hyperparameter distributions
#   C: 1e-3 .. 1e3
#   gamma: 0.01 .. 1
# -------------------------------
svm_param_distributions = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-2, 1),
    "svm__kernel": ["rbf"],
}

svm_n_iter = 25
logging.info("Starting SVM RandomizedSearchCV with %d iterations...", svm_n_iter)

# -------------------------------
# RandomizedSearchCV setup
# -------------------------------
svm_search = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=svm_param_distributions,
    n_iter=svm_n_iter,
    cv=cv_strategy,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=2,
)

# -------------------------------
# Fit on training data (X_train, y_train خام)
# -------------------------------
svm_search.fit(X_train, y_train)
logging.info("SVM RandomizedSearchCV complete.")

# Validation (cross-validated) accuracy
logging.info("Best SVM parameters found: %s", svm_search.best_params_)
logging.info(
    "Best cross-validated accuracy (mean validation accuracy over folds): %.4f",
    svm_search.best_score_,
)

# -------------------------------
# Log all tried configurations
# -------------------------------
cv_results_ = svm_search.cv_results_
for i in range(len(cv_results_["params"])):
    params = cv_results_["params"][i]
    mean_score = cv_results_["mean_test_score"][i]
    std_score = cv_results_["std_test_score"][i]
    rank = cv_results_["rank_test_score"][i]
    logging.info(
        "Config %03d | params=%s | mean_val_acc=%.4f | std_val_acc=%.4f | rank=%d",
        i,
        params,
        mean_score,
        std_score,
        rank,
    )

# -------------------------------
# Best model (Pipeline: scaler + svm)
# -------------------------------
best_svm = svm_search.best_estimator_

# -------------------------------
# Train accuracy
# -------------------------------
y_train_pred = best_svm.predict(X_train)   # خود pipeline اول نرمال می‌کند، بعد predict
train_acc = accuracy_score(y_train, y_train_pred)
logging.info("Train accuracy: %.4f", train_acc)

# -------------------------------
# Test accuracy + report + confusion matrix
# -------------------------------
y_test_pred = best_svm.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)
logging.info("Test accuracy: %.4f", test_acc)

svm_report = classification_report(y_test, y_test_pred)
svm_cm = confusion_matrix(y_test, y_test_pred)

logging.info("Tuned SVM classification report:\n%s", svm_report)
logging.info("Tuned SVM confusion matrix:\n%s", svm_cm)

# -------------------------------
# Save model
# -------------------------------
joblib.dump(best_svm, "svm_tuned_loguniform_scaled.pkl")
logging.info("Saved tuned SVM model to svm_tuned_loguniform_scaled.pkl")


2025-12-03 20:49:39,556 - INFO - Starting SVM RandomizedSearchCV with 25 iterations...


Fitting 5 folds for each of 25 candidates, totalling 125 fits


2025-12-03 20:57:21,624 - INFO - SVM RandomizedSearchCV complete.
2025-12-03 20:57:21,628 - INFO - Best SVM parameters found: {'svm__C': np.float64(4.418441521199722), 'svm__gamma': np.float64(0.021930485556643693), 'svm__kernel': 'rbf'}
2025-12-03 20:57:21,628 - INFO - Best cross-validated accuracy (mean validation accuracy over folds): 0.4945
2025-12-03 20:57:21,629 - INFO - Config 000 | params={'svm__C': np.float64(0.1767016940294795), 'svm__gamma': np.float64(0.7969454818643931), 'svm__kernel': 'rbf'} | mean_val_acc=0.1857 | std_val_acc=0.0000 | rank=21
2025-12-03 20:57:21,629 - INFO - Config 001 | params={'svm__C': np.float64(24.658329458549105), 'svm__gamma': np.float64(0.15751320499779725), 'svm__kernel': 'rbf'} | mean_val_acc=0.4505 | std_val_acc=0.0053 | rank=7
2025-12-03 20:57:21,629 - INFO - Config 002 | params={'svm__C': np.float64(0.008632008168602538), 'svm__gamma': np.float64(0.020511104188433976), 'svm__kernel': 'rbf'} | mean_val_acc=0.2880 | std_val_acc=0.0030 | rank=1

## Section 5: Decision Tree training + tuning + evaluation

In [21]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score  # اگر قبلاً ایمپورت نکردی

dt_param_distributions = {
    "max_depth": [5, 10, 20, None],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 5, 10],
}

dt_n_iter = 16
logging.info(
    "Running Decision Tree RandomizedSearchCV (n_iter=%d, random_state=42) over %d candidate values.",
    dt_n_iter,
    len(dt_param_distributions["max_depth"])
    * len(dt_param_distributions["criterion"])
    * len(dt_param_distributions["min_samples_split"]),
)

dt_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=dt_param_distributions,
    n_iter=dt_n_iter,
    random_state=42,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
logging.info("Fitting Decision Tree randomized search...")
dt_search.fit(X_train, y_train)
logging.info("Decision Tree RandomizedSearchCV complete.")

best_dt = dt_search.best_estimator_
logging.info("Best Decision Tree parameters: %s", dt_search.best_params_)
logging.info("Best cross-validated accuracy: %.4f", dt_search.best_score_)

# -------------------------------
# دقت روی TRAIN و TEST برای مدل تیون‌شده
# -------------------------------
dt_train_pred = best_dt.predict(X_train)
dt_test_pred = best_dt.predict(X_test)

dt_train_acc = accuracy_score(y_train, dt_train_pred)
dt_test_acc = accuracy_score(y_test, dt_test_pred)

logging.info("Decision Tree TRAIN accuracy: %.4f", dt_train_acc)
logging.info("Decision Tree TEST  accuracy: %.4f", dt_test_acc)

# -------------------------------
# گزارش و ماتریس سردرگمی روی داده‌های تست
# -------------------------------
dt_tuned_report = classification_report(y_test, dt_test_pred)
logging.info("Decision Tree classification report:\n%s", dt_tuned_report)

dt_tuned_cm = confusion_matrix(y_test, dt_test_pred, labels=labels)
dt_tuned_cm_df = pd.DataFrame(
    dt_tuned_cm,
    index=[f"true_{lbl}" for lbl in labels],
    columns=[f"pred_{lbl}" for lbl in labels]
)
logging.info("Decision Tree confusion matrix:\n%s", dt_tuned_cm_df)


2025-12-03 21:17:12,729 - INFO - Running Decision Tree RandomizedSearchCV (n_iter=16, random_state=42) over 24 candidate values.
2025-12-03 21:17:12,732 - INFO - Fitting Decision Tree randomized search...


Fitting 5 folds for each of 16 candidates, totalling 80 fits


2025-12-03 21:17:21,902 - INFO - Decision Tree RandomizedSearchCV complete.
2025-12-03 21:17:21,902 - INFO - Best Decision Tree parameters: {'min_samples_split': 10, 'max_depth': 10, 'criterion': 'gini'}
2025-12-03 21:17:21,902 - INFO - Best cross-validated accuracy: 0.3595
2025-12-03 21:17:21,909 - INFO - Decision Tree TRAIN accuracy: 0.4493
2025-12-03 21:17:21,909 - INFO - Decision Tree TEST  accuracy: 0.3604
2025-12-03 21:17:21,913 - INFO - Decision Tree classification report:
              precision    recall  f1-score   support

           0       0.59      0.45      0.51       422
           1       0.31      0.15      0.20       334
           2       0.32      0.47      0.38       620
           3       0.26      0.21      0.23       373
           4       0.35      0.41      0.38       973
           5       0.42      0.11      0.17       289
           6       0.25      0.47      0.32       525
           7       0.26      0.16      0.20       364
           8       0.53     

## Section 6: Save models
Persist tuned models for reuse.

In [19]:

svm_model_path = Path("svm_phase3.pkl")
dt_model_path = Path("dt_phase3.pkl")

logging.info("Saving tuned models to disk...")
joblib.dump(best_svm, svm_model_path)
joblib.dump(best_dt, dt_model_path)

logging.info("Saved tuned SVM model to: %s", svm_model_path.resolve())
logging.info("Saved tuned Decision Tree model to: %s", dt_model_path.resolve())


2025-12-03 20:58:15,997 - INFO - Saving tuned models to disk...
2025-12-03 20:58:16,003 - INFO - Saved tuned SVM model to: E:\university\ML- fazl\project2\svm_phase3.pkl
2025-12-03 20:58:16,004 - INFO - Saved tuned Decision Tree model to: E:\university\ML- fazl\project2\dt_phase3.pkl
